# 78 — Soft Label Training on Intermediate TL (Target: beat 0.521 val-tuned)

**Motivasi:** nb 71 (CNN TL + KL-div soft) = **0.517** (+0.09 vs hard). nb 72 (Late Fusion TL + soft) = **0.437** (gagal — soft label tidak transfer ke weighted softmax averaging).

**Hipotesis:** Intermediate Fusion TL cocok untuk soft label karena **joint training**: gradient dari soft target backprop **langsung** ke both branches (image + landmark) via shared fusion head, berbeda dengan Late Fusion yang train branch terpisah lalu merge di softmax level.

**Target:** beat **Intermediate TL 4c B3 val-tuned = 0.521** (current best overall) atau setidaknya beat Intermediate TL 4c B1 hard baseline.

**Setup:** 4-class Primer conf60, **B1 baseline** (no class weights, no augmentation) — match methodology nb 71/72 (isolate efek loss function). 4 loss variants.

**4 Configs:**
| # | Loss | Target | Tujuan |
|---|------|--------|--------|
| A | Hard CE | one-hot | baseline (ref: Intermediate TL 4c B1 = 0.489 val-tuned) |
| B | Soft CE | Face API distribusi | alternative soft loss |
| C | KL-divergence | Face API distribusi | **winning loss nb 71** |
| D | Label Smoothing ε=0.1 | hard + uniform | artificial smoothing baseline |

In [ ]:
import sys, os, json
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from training.models import IntermediateFusionTransfer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

DATA_DIR   = PROJECT_ROOT / "data" / "dataset_frontonly_conf60"
OUTPUT_DIR = PROJECT_ROOT / "models" / "frontonly_conf60" / "soft_label"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 4
BATCH_SIZE  = 32
EPOCHS      = 50      # align dengan nb 71 (hyperparam konsisten lintas arch)
PATIENCE    = 15
LR_TL       = 5e-5    # align dengan nb 71
SEED        = 42

torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

REMAP_4 = np.array([0, 1, 2, 3, 3, 3, 3], dtype=np.int64)
EMOTIONS = ["neutral", "happy", "sad", "negative"]

In [ ]:
# ── Load data (4-class + soft labels aggregated) ──
def remap_soft_to_4class(y_soft_7):
    y4 = np.zeros((len(y_soft_7), 4), dtype=np.float32)
    y4[:, 0] = y_soft_7[:, 0]
    y4[:, 1] = y_soft_7[:, 1]
    y4[:, 2] = y_soft_7[:, 2]
    y4[:, 3] = y_soft_7[:, 3:7].sum(axis=1)
    return y4

def load_split(split):
    img = np.load(DATA_DIR / f'X_{split}_images.npy')
    lm  = np.load(DATA_DIR / f'X_{split}_landmarks.npy')
    y7  = np.load(DATA_DIR / f'y_{split}.npy')
    ys7 = np.load(DATA_DIR / f'y_{split}_soft.npy')
    return img, lm, REMAP_4[y7], remap_soft_to_4class(ys7)

X_tr_img, X_tr_lm, y_tr, y_tr_soft = load_split('train')
X_v_img,  X_v_lm,  y_v,  y_v_soft  = load_split('val')
X_te_img, X_te_lm, y_te, y_te_soft = load_split('test')

print(f'Train: img={X_tr_img.shape}  lm={X_tr_lm.shape}  y_hard={y_tr.shape}  y_soft={y_tr_soft.shape}')
print(f'Val:   img={X_v_img.shape}   y_hard={y_v.shape}')
print(f'Test:  img={X_te_img.shape}  y_hard={y_te.shape}')
print(f'\n4-class hard dist (train): {np.bincount(y_tr, minlength=4).tolist()}')
print(f'Soft label sum check: mean={y_tr_soft.sum(axis=1).mean():.4f} min={y_tr_soft.sum(axis=1).min():.4f}')

In [ ]:
# ── Dataset + loaders (fusion: image + landmark + y_hard + y_soft) ──
class SoftFusionDS(Dataset):
    def __init__(self, images, landmarks, y_hard, y_soft):
        self.images = images
        self.landmarks = torch.from_numpy(landmarks).float()
        self.y_hard = torch.from_numpy(y_hard).long()
        self.y_soft = torch.from_numpy(y_soft).float()
    def __len__(self): return len(self.y_hard)
    def __getitem__(self, i):
        img = torch.from_numpy(self.images[i]).permute(2, 0, 1).contiguous()
        return img, self.landmarks[i], self.y_hard[i], self.y_soft[i]

tr_ds = SoftFusionDS(X_tr_img, X_tr_lm, y_tr, y_tr_soft)
v_ds  = SoftFusionDS(X_v_img,  X_v_lm,  y_v,  y_v_soft)
te_ds = SoftFusionDS(X_te_img, X_te_lm, y_te, y_te_soft)

tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
v_loader  = DataLoader(v_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
te_loader = DataLoader(te_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Loaders ready — batches: tr={len(tr_loader)}, v={len(v_loader)}, te={len(te_loader)}')

In [ ]:
# ── Loss functions (4 variants) ──
def hard_ce_loss(output, y_h, y_s):
    return F.cross_entropy(output, y_h)

def soft_ce_loss(output, y_h, y_s):
    log_probs = F.log_softmax(output, dim=1)
    return -(y_s * log_probs).sum(dim=1).mean()

def kl_div_loss(output, y_h, y_s):
    log_probs = F.log_softmax(output, dim=1)
    return F.kl_div(log_probs, y_s, reduction='batchmean')

def label_smooth_loss(output, y_h, y_s, eps=0.1, K=NUM_CLASSES):
    """(1-ε)·one_hot + ε/K uniform, CE with smoothed target."""
    one_hot = F.one_hot(y_h, K).float()
    smoothed = (1 - eps) * one_hot + eps / K
    log_probs = F.log_softmax(output, dim=1)
    return -(smoothed * log_probs).sum(dim=1).mean()

LOSS_FNS = {
    'A_hard_CE':         hard_ce_loss,
    'B_soft_CE':         soft_ce_loss,
    'C_KL_div':          kl_div_loss,
    'D_label_smooth':    label_smooth_loss,
}

In [ ]:
# ── Training + evaluation ──
def train_intermediate_tl(loss_fn, save_path):
    model = IntermediateFusionTransfer(num_classes=NUM_CLASSES).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=LR_TL)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='max', factor=0.5, patience=8, min_lr=1e-7)

    best_vf1, best_ep, stale = 0.0, 0, 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for img, lm, y_h, y_s in tr_loader:
            img, lm, y_h, y_s = img.to(device), lm.to(device), y_h.to(device), y_s.to(device)
            out = model(img, lm)
            loss = loss_fn(out, y_h, y_s)
            opt.zero_grad(); loss.backward(); opt.step()

        # Val — always report hard-label Macro F1 (argmax)
        model.eval()
        yh, yp = [], []
        with torch.no_grad():
            for img, lm, y_h, _ in v_loader:
                img, lm = img.to(device), lm.to(device)
                p = model(img, lm).argmax(1).cpu().numpy()
                yh.append(y_h.numpy()); yp.append(p)
        vf1 = f1_score(np.concatenate(yh), np.concatenate(yp), average='macro', zero_division=0)
        sched.step(vf1)

        if vf1 > best_vf1:
            best_vf1, best_ep, stale = vf1, epoch, 0
            torch.save(model.state_dict(), save_path)
        else:
            stale += 1
            if stale >= PATIENCE:
                print(f'  Early stop @ epoch {epoch} (best={best_vf1:.4f} @ {best_ep})')
                break
        if epoch % 10 == 0:
            print(f'  epoch {epoch}: val_f1={vf1:.4f} best={best_vf1:.4f}@{best_ep}')

    return best_vf1, best_ep

def evaluate_test(ckpt_path):
    model = IntermediateFusionTransfer(num_classes=NUM_CLASSES).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    model.eval()
    yh, yp = [], []
    with torch.no_grad():
        for img, lm, y_h, _ in te_loader:
            img, lm = img.to(device), lm.to(device)
            p = model(img, lm).argmax(1).cpu().numpy()
            yh.append(y_h.numpy()); yp.append(p)
    yt, yp = np.concatenate(yh), np.concatenate(yp)
    return {
        'accuracy':     float(accuracy_score(yt, yp)),
        'macro_f1':     float(f1_score(yt, yp, average='macro', zero_division=0)),
        'micro_f1':     float(f1_score(yt, yp, average='micro', zero_division=0)),
        'weighted_f1':  float(f1_score(yt, yp, average='weighted', zero_division=0)),
        'confusion_matrix': confusion_matrix(yt, yp, labels=list(range(NUM_CLASSES))).tolist(),
        'classification_report': classification_report(yt, yp, target_names=EMOTIONS, zero_division=0, output_dict=True),
    }

## Run 4 Configs

In [ ]:
results = {}

for cfg_name, loss_fn in LOSS_FNS.items():
    print(f"\n{'='*70}\n  Config: {cfg_name}\n{'='*70}")
    save_dir = OUTPUT_DIR / f'{NUM_CLASSES}c' / f'Intermediate_TL_{cfg_name}'
    save_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = save_dir / 'model.pth'

    best_vf1, best_ep = train_intermediate_tl(loss_fn, str(ckpt_path))
    test_metrics = evaluate_test(str(ckpt_path))

    print(f'  {cfg_name}: val_f1={best_vf1:.4f}@ep{best_ep}  '
          f'test_f1={test_metrics["macro_f1"]:.4f}  acc={test_metrics["accuracy"]:.4f}')

    results[cfg_name] = {
        **test_metrics,
        'val_macro_f1': float(best_vf1),
        'best_epoch': int(best_ep),
    }

out_json = OUTPUT_DIR / 'soft_intermediate_tl_4c_results.json'
with open(out_json, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved: {out_json}')

## Comparison vs Existing Baselines

In [ ]:
print(f"\n{'='*86}")
print(f'  Soft Label Intermediate TL vs Baselines (4-class, Primer conf60 test)')
print(f"{'='*86}")
print(f"  {'Config':<38} {'Macro':>8} {'Micro':>8} {'Weighted':>10} {'Acc':>8}")
print(f"  {'-'*78}")

baselines = [
    ('Intermediate TL 4c B1 hard (existing)',   0.489, 0.824, 0.813, 0.824),
    ('Intermediate TL 4c B3 hard (val-tuned ★)', 0.521, 0.822, 0.828, 0.822),
    ('CNN TL 4c B1 soft KL (nb 71)',             0.517, 0.821, 0.826, 0.821),
    ('Late Fusion TL 4c B3 soft KL (nb 72)',     0.437, 0.809, 0.801, 0.809),
]
for name, m, mi, w, a in baselines:
    print(f"  {name:<38} {m:>8.4f} {mi:>8.4f} {w:>10.4f} {a:>8.4f}")
print(f"  {'-'*78}")

label_map = {'A_hard_CE': 'A Hard CE', 'B_soft_CE': 'B Soft CE',
             'C_KL_div':  'C KL-divergence', 'D_label_smooth': 'D Label Smooth ε=0.1'}
for cfg, label in label_map.items():
    r = results[cfg]
    marker = ' ★' if r['macro_f1'] == max(results[c]['macro_f1'] for c in results) else ''
    print(f"  Intermediate TL {label:<24} {r['macro_f1']:>8.4f} {r['micro_f1']:>8.4f} "
          f"{r['weighted_f1']:>10.4f} {r['accuracy']:>8.4f}{marker}")

print(f"\n  Target overall best: 0.521 (Intermediate TL B3 val-tuned)")
print(f"  Target soft label: 0.517 (CNN TL 4c B1 soft KL from nb 71)")

## Analysis

**Ekspektasi (berdasarkan pattern nb 71):**
- C KL-div ≥ B Soft CE > D Label Smoothing > A Hard CE
- Gain soft label vs hard: +0.05-0.10 (konsisten nb 71)
- **Target konkret:** C KL-div > 0.521 → soft label + feature-level fusion = novel SOTA untuk Primer 4c

**Kalau gagal beat 0.521:**
- Intermediate fusion head mungkin saturated (soft signal hilang di 2-layer MLP setelah concat)
- Alternatif: try with class weights (B2-style) + soft — tapi perlu hati-hati karena CE weight tidak apply ke soft target secara natural

**Next step setelah hasil:**
- Kalau C tembus 0.521 → update paper best, add caveat methodology di Section 4
- Kalau gagal → disclose di Discussion; soft label finding di single-modal (nb 71) tetap valuable sebagai eksplorasi, bukan SOTA push

**Commit results (di VPS):**
```bash
git add models/frontonly_conf60/soft_label/4c/Intermediate_TL_*/ \
        models/frontonly_conf60/soft_label/soft_intermediate_tl_4c_results.json \
        notebooks/results/78_*
git commit -m "Add Soft Label Intermediate TL results (nb 78)"
```